# 01 — Download & Explore Multi-VSL Dataset

**Vietnamese Sign Language Recognition using VideoMAEv2**

Notebook này:
1. Download Multi-VSL dataset (50 classes, frontal view) từ Google Drive
2. Visualize video samples
3. Phân tích thống kê dataset
4. Tạo train/val split

**Dataset**: [Multi-VSL (WACV 2025)](https://github.com/Etdihatthoc/Multi-VSL_WACV_2025) — 84,000+ videos, 1000 glosses, 30 signers

**Chạy trên**: Google Colab / Kaggle (GPU T4) hoặc Local

In [ ]:
# ============================================================
# CẤU HÌNH — Thay đổi ở đây nếu cần
# ============================================================

NUM_CLASSES = 50          # Số lượng classes để download (tối đa 1000)
NUM_FRAMES = 16           # Số frames mỗi video (input cho VideoMAE)
VAL_RATIO = 0.2           # Tỉ lệ validation set
VIEW = "frontal"          # Chỉ dùng frontal view (matched với webcam inference)
SEED = 42

# Google Drive folder ID của Multi-VSL dataset
# Folder: https://drive.google.com/drive/folders/1yUU1m2hy_CjaXDDoR_6i9Y3T1XL2pD4C
DRIVE_FOLDER_ID = "1yUU1m2hy_CjaXDDoR_6i9Y3T1XL2pD4C"

## 1. Setup Environment

In [ ]:
import os
import sys

# === Detect environment ===
def detect_environment():
    """Detect if running on Colab, Kaggle, or Local."""
    try:
        import google.colab
        return "colab"
    except ImportError:
        pass
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_environment()
print(f"🖥️ Environment: {ENV}")

# === Set up paths ===
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/vsl-recognition"
    os.makedirs(BASE_DIR, exist_ok=True)
elif ENV == "kaggle":
    BASE_DIR = "/kaggle/working/vsl-recognition"
    os.makedirs(BASE_DIR, exist_ok=True)
else:
    # Local: use project root
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(BASE_DIR, "data", "multi_vsl")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"📁 Base dir: {BASE_DIR}")
print(f"📁 Data dir: {DATA_DIR}")
print(f"📁 Model dir: {MODEL_DIR}")

In [ ]:
# === Install dependencies (Colab/Kaggle) ===
if ENV in ("colab", "kaggle"):
    !pip install -q gdown decord transformers accelerate torch torchvision

import glob
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, display

random.seed(SEED)
np.random.seed(SEED)
print("✅ Dependencies ready")

## 2. Download Multi-VSL Dataset

**Multi-VSL (WACV 2025)** dataset structure trên Google Drive:
```
Multi-VSL/
├── frontal/           ← Chúng ta chỉ dùng view này
│   ├── class_001/
│   │   ├── signer01_001.avi
│   │   ├── signer02_001.avi
│   │   └── ...
│   ├── class_002/
│   └── ... (1000 classes)
├── right/
└── left/
```

⚠️ **LƯU Ý**: Dataset ~2GB cho frontal view. Download lần đầu mất ~10-30 phút tùy tốc độ mạng.

Nếu download tự động không hoạt động, bạn có thể:
1. Vào [Google Drive link](https://drive.google.com/drive/folders/1yUU1m2hy_CjaXDDoR_6i9Y3T1XL2pD4C)
2. Download thủ công thư mục frontal
3. Upload lên Colab/Kaggle vào đúng `DATA_DIR`

In [ ]:
import gdown
import shutil
import re

def parse_video_filename(filename: str):
    """Parse Multi-VSL video filename to extract class and view info.
    
    Filename pattern:
      {class_id}_{class_name}_{range}_{devices}_{id}___{view}_{device}_{signer}_{view2}_{ord}_{num}.mp4
    Example:
      01_Co-Hien_1-100_1-2-3_0108___center_device02_signer01_center_ord1_61.mp4
    
    Returns: (class_name, view) or (None, None) if parsing fails.
    """
    stem = Path(filename).stem
    
    # Split by triple underscore to separate class info from recording info
    parts = stem.split("___")
    if len(parts) != 2:
        return None, None
    
    class_info = parts[0]  # e.g. "01_Co-Hien_1-100_1-2-3_0108"
    recording_info = parts[1]  # e.g. "center_device02_signer01_center_ord1_61"
    
    # Extract class: use the full class_info as class name
    # This groups all videos of the same sign together
    class_name = class_info
    
    # Extract view from recording info (first token)
    view = recording_info.split("_")[0]  # "center", "left", or "right"
    
    return class_name, view


def download_multi_vsl(drive_folder_id: str, output_dir: str, num_classes: int = 50):
    """Download Multi-VSL dataset from Google Drive.
    
    After download, filter center-view (frontal) videos only and organize
    into output_dir/class_name/video.mp4 format.
    """
    output_path = Path(output_dir)
    
    # Check if already downloaded and organized
    existing_classes = [d for d in output_path.iterdir() if d.is_dir()] if output_path.exists() else []
    if len(existing_classes) >= num_classes:
        print(f"✅ Dataset already exists with {len(existing_classes)} classes")
        return output_path
    
    print(f"📥 Downloading Multi-VSL dataset ({num_classes} classes, center/frontal view)...")
    print(f"   Source: https://drive.google.com/drive/folders/{drive_folder_id}")
    print(f"   Target: {output_dir}")
    print(f"   ⏳ This may take 10-30 minutes depending on connection speed...")
    
    # Download entire folder into a temp location
    download_dir = output_path.parent / "multi_vsl_download"
    os.makedirs(download_dir, exist_ok=True)
    
    try:
        gdown.download_folder(
            id=drive_folder_id,
            output=str(download_dir),
            quiet=False,
            remaining_ok=True,
        )
        print("✅ Download complete!")
    except Exception as e:
        print(f"⚠️ Auto-download failed: {e}")
        print(f"\n📋 HƯỚNG DẪN DOWNLOAD THỦ CÔNG:")
        print(f"   1. Vào: https://drive.google.com/drive/folders/{drive_folder_id}")
        print(f"   2. Download toàn bộ dataset")
        print(f"   3. Giải nén vào: {output_dir}")
        print(f"   4. Đảm bảo cấu trúc: {output_dir}/class_name/video.mp4")
        return None
    
    # === Parse filenames and organize ===
    print("\n🔍 Scanning downloaded files...")
    
    all_videos = list(download_dir.rglob("*.avi")) + list(download_dir.rglob("*.mp4"))
    print(f"   Found {len(all_videos)} video files total")
    
    if not all_videos:
        print("⚠️ No video files found after download!")
        print(f"   Contents of {download_dir}:")
        for item in sorted(download_dir.rglob("*"))[:20]:
            print(f"     {item.relative_to(download_dir)}")
        return None
    
    # Try filename-based parsing first (Multi-VSL flat structure)
    # Pattern: {class}___{view}_{device}_{signer}_{view2}_{ord}_{num}.mp4
    parsed_videos = []
    unparsed_videos = []
    views_found = set()
    
    for vp in all_videos:
        class_name, view = parse_video_filename(vp.name)
        if class_name and view:
            parsed_videos.append((vp, class_name, view))
            views_found.add(view)
        else:
            unparsed_videos.append(vp)
    
    if parsed_videos:
        # Filename-based structure detected
        print(f"   📋 Parsed {len(parsed_videos)} filenames successfully")
        print(f"   Views found: {sorted(views_found)}")
        
        # Filter to center view (= frontal/trực diện)
        center_videos = [(vp, cls) for vp, cls, view in parsed_videos if view == "center"]
        
        if not center_videos:
            # Fallback: try frontal or first view found
            for try_view in ["frontal", "front", sorted(views_found)[0]]:
                center_videos = [(vp, cls) for vp, cls, view in parsed_videos if view == try_view]
                if center_videos:
                    print(f"   Using view: '{try_view}'")
                    break
        else:
            print(f"   Filtered to {len(center_videos)} center-view videos")
            print(f"   (excluded {len(parsed_videos) - len(center_videos)} left/right view videos)")
        
        # Group by class
        class_videos = {}
        for vp, cls in center_videos:
            if cls not in class_videos:
                class_videos[cls] = []
            class_videos[cls].append(vp)
    else:
        # Fallback: folder-based structure
        print("   ⚠️ Could not parse filenames, falling back to folder structure")
        # Filter frontal view by folder name
        frontal = [v for v in all_videos if "frontal" in [p.lower() for p in v.parts]]
        if frontal:
            all_videos = frontal
        
        class_videos = {}
        for vp in all_videos:
            class_name = vp.parent.name
            if class_name not in class_videos:
                class_videos[class_name] = []
            class_videos[class_name].append(vp)
    
    print(f"   Found {len(class_videos)} unique classes")
    
    if not class_videos:
        print("⚠️ No classes found! Check dataset structure.")
        print(f"   Sample files: {[v.name for v in all_videos[:5]]}")
        return None
    
    # Select first N classes (sorted)
    sorted_classes = sorted(class_videos.keys())[:num_classes]
    print(f"   Selecting {len(sorted_classes)} classes for training")
    
    # Move videos into organized structure
    os.makedirs(output_path, exist_ok=True)
    total_moved = 0
    for cls_name in sorted_classes:
        cls_dir = output_path / cls_name
        os.makedirs(cls_dir, exist_ok=True)
        for vp in class_videos[cls_name]:
            dest = cls_dir / vp.name
            if not dest.exists():
                shutil.move(str(vp), str(dest))
                total_moved += 1
    
    print(f"\n✅ Organized {total_moved} videos into {len(sorted_classes)} class folders")
    print(f"   Structure: {output_dir}/class_name/video.mp4")
    
    # Show first few classes
    for cls_name in sorted_classes[:5]:
        n = len(list((output_path / cls_name).iterdir()))
        print(f"   {cls_name}: {n} videos")
    if len(sorted_classes) > 5:
        print(f"   ... and {len(sorted_classes) - 5} more classes")
    
    # Clean up temporary download directory
    shutil.rmtree(str(download_dir), ignore_errors=True)
    print(f"   🧹 Cleaned up temp download directory")
    
    return output_path

# Download and organize
dataset_path = download_multi_vsl(DRIVE_FOLDER_ID, DATA_DIR, NUM_CLASSES)


In [ ]:
# === Explore dataset structure ===
def explore_dataset(data_dir: str, num_classes: int = 50):
    """Explore and report dataset statistics."""
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print("⚠️ Data directory not found. Please download first.")
        return None
    
    # Find all class directories
    all_classes = sorted([d for d in data_path.rglob("*") if d.is_dir() and any(f.suffix.lower() in ('.avi', '.mp4') for f in d.iterdir())])
    
    if not all_classes:
        # Try one level deeper
        for subdir in data_path.iterdir():
            if subdir.is_dir():
                sub_classes = sorted([d for d in subdir.iterdir() if d.is_dir()])
                if sub_classes:
                    all_classes = sub_classes
                    break
    
    if not all_classes:
        print("⚠️ No video class directories found!")
        print(f"   Contents of {data_dir}: {list(data_path.iterdir())}")
        return None
    
    # Select first N classes
    selected_classes = all_classes[:num_classes]
    
    # Statistics
    stats = []
    total_videos = 0
    for cls_dir in selected_classes:
        videos = [f for f in cls_dir.iterdir() if f.suffix.lower() in ('.avi', '.mp4', '.mov', '.mkv')]
        stats.append({
            "class": cls_dir.name,
            "num_videos": len(videos),
            "path": str(cls_dir)
        })
        total_videos += len(videos)
    
    print(f"📊 Dataset Statistics:")
    print(f"   Total classes found: {len(all_classes)}")
    print(f"   Selected classes: {len(selected_classes)}")
    print(f"   Total videos (selected): {total_videos}")
    print(f"   Avg videos/class: {total_videos / len(selected_classes):.1f}")
    print(f"   Min videos/class: {min(s['num_videos'] for s in stats)}")
    print(f"   Max videos/class: {max(s['num_videos'] for s in stats)}")
    print(f"\
   First 10 classes:")
    for s in stats[:10]:
        print(f"     {s['class']}: {s['num_videos']} videos")
    
    return stats, selected_classes

result = explore_dataset(DATA_DIR, NUM_CLASSES)

## 3. Visualize Video Samples

Xem trực tiếp video trong notebook để verify dataset hoạt động đúng.

In [ ]:
import cv2
from IPython.display import Image as IPImage
import io
from PIL import Image

def visualize_video_frames(video_path: str, num_frames: int = 8, figsize=(16, 4)):
    """Display uniform-sampled frames from a video."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    
    if not frames:
        print(f"⚠️ Cannot read: {video_path}")
        return
    
    total = len(frames)
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    sampled = [frames[i] for i in indices]
    
    fig, axes = plt.subplots(1, num_frames, figsize=figsize)
    fig.suptitle(f"{Path(video_path).parent.name} — {total} frames total", fontsize=12)
    for ax, frame, idx in zip(axes, sampled, indices):
        ax.imshow(frame)
        ax.set_title(f"Frame {idx}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_sample_videos(data_dir: str, num_samples: int = 3):
    """Show random video samples from the dataset."""
    data_path = Path(data_dir)
    all_videos = list(data_path.rglob("*.avi")) + list(data_path.rglob("*.mp4"))
    
    if not all_videos:
        print("⚠️ No videos found. Please check dataset download.")
        return
    
    samples = random.sample(all_videos, min(num_samples, len(all_videos)))
    
    print(f"🎬 Showing {len(samples)} random video samples:\
")
    for video_path in samples:
        print(f"  Class: {video_path.parent.name}")
        print(f"  File: {video_path.name}")
        visualize_video_frames(str(video_path))
        print()

show_sample_videos(DATA_DIR, num_samples=3)

## 4. Dataset Statistics & Distribution

In [ ]:
def plot_dataset_stats(data_dir: str, num_classes: int = 50):
    """Plot video count distribution and video duration stats."""
    data_path = Path(data_dir)
    all_videos = list(data_path.rglob("*.avi")) + list(data_path.rglob("*.mp4"))
    
    if not all_videos:
        print("⚠️ No videos found.")
        return
    
    # Count per class
    class_counts = {}
    for v in all_videos:
        cls_name = v.parent.name
        class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
    
    # Sort and select top N
    sorted_classes = sorted(class_counts.items(), key=lambda x: x[0])[:num_classes]
    classes = [c[0] for c in sorted_classes]
    counts = [c[1] for c in sorted_classes]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Distribution of videos per class
    axes[0].bar(range(len(counts)), counts, color='steelblue', alpha=0.7)
    axes[0].axhline(y=np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
    axes[0].set_xlabel('Class Index')
    axes[0].set_ylabel('Number of Videos')
    axes[0].set_title(f'Videos per Class ({len(sorted_classes)} classes)')
    axes[0].legend()
    
    # Video duration distribution (sample)
    durations = []
    sample_videos = random.sample(all_videos, min(100, len(all_videos)))
    for v in sample_videos:
        cap = cv2.VideoCapture(str(v))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if frame_count > 0 and fps > 0:
            durations.append(frame_count / fps)
    
    if durations:
        axes[1].hist(durations, bins=20, color='coral', alpha=0.7, edgecolor='black')
        axes[1].axvline(x=np.mean(durations), color='red', linestyle='--', label=f'Mean: {np.mean(durations):.2f}s')
        axes[1].set_xlabel('Duration (seconds)')
        axes[1].set_ylabel('Count')
        axes[1].set_title(f'Video Duration Distribution (sample of {len(durations)})')
        axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\
📊 Summary:")
    print(f"   Total classes: {len(class_counts)}")
    print(f"   Total videos: {len(all_videos)}")
    if durations:
        print(f"   Avg duration: {np.mean(durations):.2f}s")
        print(f"   Duration range: {min(durations):.2f}s - {max(durations):.2f}s")

plot_dataset_stats(DATA_DIR, NUM_CLASSES)

## 5. Create Train/Val Split & Save Metadata

Split theo signer (nếu filename chứa signer ID) để tránh data leakage.

In [ ]:
def create_split(data_dir: str, num_classes: int = 50, val_ratio: float = 0.2, seed: int = 42):
    """Create train/val split and save as JSON metadata."""
    data_path = Path(data_dir)
    all_videos = sorted(list(data_path.rglob("*.avi")) + list(data_path.rglob("*.mp4")))
    
    if not all_videos:
        print("⚠️ No videos found.")
        return None
    
    # Get unique classes
    class_dirs = sorted(set(v.parent for v in all_videos))
    selected_dirs = class_dirs[:num_classes]
    class_names = [d.name for d in selected_dirs]
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    
    # Collect videos for selected classes
    videos_by_class = {name: [] for name in class_names}
    for v in all_videos:
        cls = v.parent.name
        if cls in class_to_idx:
            videos_by_class[cls].append(str(v))
    
    # Split
    random.seed(seed)
    train_data = []
    val_data = []
    
    for cls_name, videos in videos_by_class.items():
        random.shuffle(videos)
        split_idx = max(1, int(len(videos) * (1 - val_ratio)))
        train_videos = videos[:split_idx]
        val_videos = videos[split_idx:]
        
        for v in train_videos:
            train_data.append({"path": v, "label": class_to_idx[cls_name], "class": cls_name})
        for v in val_videos:
            val_data.append({"path": v, "label": class_to_idx[cls_name], "class": cls_name})
    
    # Save metadata
    metadata = {
        "num_classes": len(class_names),
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "train_size": len(train_data),
        "val_size": len(val_data),
        "num_frames": NUM_FRAMES,
        "view": VIEW,
    }
    
    meta_dir = Path(data_dir).parent
    
    with open(meta_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    with open(meta_dir / "train.json", "w") as f:
        json.dump(train_data, f, indent=2)
    with open(meta_dir / "val.json", "w") as f:
        json.dump(val_data, f, indent=2)
    
    print(f"✅ Split created:")
    print(f"   Classes: {len(class_names)}")
    print(f"   Train: {len(train_data)} videos")
    print(f"   Val: {len(val_data)} videos")
    print(f"   Saved to: {meta_dir}")
    print(f"\
   Class names (first 10): {class_names[:10]}")
    
    return metadata, train_data, val_data

result = create_split(DATA_DIR, NUM_CLASSES, VAL_RATIO, SEED)

## 6. Test Video Loading Pipeline

Kiểm tra xem pipeline load video + transform hoạt động đúng trước khi training.

In [ ]:
def test_video_loading(data_dir: str, num_frames: int = 16):
    """Test the video loading and transform pipeline."""
    data_path = Path(data_dir)
    all_videos = list(data_path.rglob("*.avi")) + list(data_path.rglob("*.mp4"))
    
    if not all_videos:
        print("⚠️ No videos to test.")
        return
    
    video_path = str(random.choice(all_videos))
    print(f"🎬 Testing video pipeline:")
    print(f"   File: {video_path}")
    
    # Load raw frames
    cap = cv2.VideoCapture(video_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    
    frames = np.array(frames)
    print(f"   Raw frames: {frames.shape}  (T, H, W, C)")
    
    # Uniform sample
    T = len(frames)
    if T >= num_frames:
        indices = np.linspace(0, T - 1, num_frames, dtype=int)
    else:
        indices = np.arange(T)
        pad = np.full(num_frames - T, T - 1, dtype=int)
        indices = np.concatenate([indices, pad])
    
    sampled = frames[indices]
    print(f"   Sampled frames: {sampled.shape}")
    
    # Transform (normalize + resize)
    import torch
    from torchvision.transforms import Resize, CenterCrop, Normalize
    
    video_tensor = torch.from_numpy(sampled).float() / 255.0
    video_tensor = video_tensor.permute(0, 3, 1, 2)  # (T, C, H, W)
    
    resize = Resize(256, antialias=True)
    crop = CenterCrop(224)
    normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    transformed = []
    for t in range(num_frames):
        frame = video_tensor[t]
        frame = resize(frame)
        frame = crop(frame)
        frame = normalize(frame)
        transformed.append(frame)
    
    final = torch.stack(transformed)
    print(f"   Final tensor: {final.shape}  (T, C, H, W)")
    print(f"   Value range: [{final.min():.3f}, {final.max():.3f}]")
    print(f"   ✅ Pipeline works! Ready for VideoMAE input.")
    
    # Visualize transform
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    fig.suptitle("Top: Raw frames | Bottom: After transform (denormalized)", fontsize=11)
    
    for i in range(8):
        idx = i * 2  # Show every other frame
        axes[0, i].imshow(sampled[idx])
        axes[0, i].axis("off")
        
        # Denormalize for visualization
        denorm = final[idx].clone()
        denorm[0] = denorm[0] * 0.229 + 0.485
        denorm[1] = denorm[1] * 0.224 + 0.456
        denorm[2] = denorm[2] * 0.225 + 0.406
        denorm = denorm.permute(1, 2, 0).clamp(0, 1).numpy()
        axes[1, i].imshow(denorm)
        axes[1, i].axis("off")
    
    plt.tight_layout()
    plt.show()

test_video_loading(DATA_DIR, NUM_FRAMES)

## Done! Next Steps

✅ Dataset downloaded và explored  
✅ Train/Val split saved (metadata.json, train.json, val.json)  
✅ Video pipeline tested  

**Tiếp theo**: Chạy notebook `02_train_videomae.ipynb` để fine-tune VideoMAEv2-Small trên dataset này.